# 18_descriptors_for_weka — descriptor 계산 & WEKA용 CSV 만들기

**한 줄 요약:** 1:1 학습셋 분자들에 **descriptor(분자를 숫자로 요약한 값) 217개**를 계산해,
(1) 전체 Excel, (2) WEKA용 CSV 두 파일을 만든다.

**용어:** descriptor=분자량·logP 같은 물성 수치 / potency=정답(1=active, 0=inactive) / WEKA=쓸모있는 descriptor를 골라주는 도구.
**큰 흐름:** ① 준비 → ② 데이터 읽기 → ③ descriptor 계산+Excel → ④ WEKA용 정제+CSV

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다. 앞 셀에서 만든 값을 뒤 셀이 쓰므로 **순서대로**.
> - 각 코드 셀은 **[① 무슨 작업인지 설명] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서로 놓았다.
>   ③은 그 셀에 **처음 나온** 함수·문법을 잘게 푼 것(이미 나온 건 반복 안 함).
> - 코드 줄 뒤 `# ...` 은 **주석**(설명)이라 실행에 영향 없음.

### 셀 1 — 준비: 폴더 위치 맞추고 도구 불러오기
프로젝트 최상위에서 실행되게 폴더를 맞추고, 계산에 필요한 라이브러리를 가져온다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

🔎 **코드 뜯어보기 (셀 1)**
- `import os` : **import**는 "남이 만든 도구 묶음(라이브러리)을 가져와라"는 명령. **os**는 폴더·파일을 다루는 파이썬 기본 도구.
- `import numpy as np` : **numpy**는 숫자·배열 계산 라이브러리. **as np**는 "앞으로 numpy를 짧게 **np**라고 부르겠다"는 별명 지정. 이후 `np.___` 로 쓴다.
- `import pandas as pd` : **pandas**는 엑셀 같은 **표(DataFrame)** 를 다루는 라이브러리. 별명 **pd**.
- `from rdkit import Chem` : **from A import B** = 큰 묶음 A(rdkit)에서 **B(Chem)만 콕 집어** 가져오기. Chem은 분자를 다루는 부분.
- `os.path.isdir('data')` : 이름 뒤 **괄호 `()`** 는 "실행". `isdir`는 'data라는 폴더가 있나?'를 True/False로 답하는 **함수**.
- `if 조건:` + 들여쓰기 : 조건이 참이면 **아래 들여쓴 줄**만 실행. 파이썬은 **들여쓰기**로 '어디에 속한 코드인지'를 표시한다.
- `os.chdir('..')` : 작업 폴더 바꾸기. `'..'` 는 '한 칸 위 폴더'.
- `print(...)` : 괄호 안 내용을 **화면에 출력**.
- `RDLogger.DisableLog('rdApp.*')` : RDKit 경고 메시지를 꺼서 화면을 깔끔하게.

### 셀 2 — 1:1 학습셋 파일 읽기
CSV를 표로 읽고, 정답 열 이름을 potency로 바꾼 뒤 분포를 확인한다.

In [ ]:
# 1:1 학습셋 로드 (active + 실측 inactive + decoy). label(1/0) -> potency
SRC = 'data/train_1to1.csv'
df = pd.read_csv(SRC)
df = df.rename(columns={'label': 'potency'})
print('[0] 1:1 학습셋 shape:', df.shape)
print('    potency 분포:', dict(df.potency.value_counts()))
print('    source 분포:', dict(df.source.value_counts()))

🔎 **코드 뜯어보기 (셀 2)**
- `SRC = 'data/train_1to1.csv'` : **`=`** 는 오른쪽 값을 왼쪽 **이름(변수)** 에 저장. 여기선 파일 경로 글자(**문자열**, 따옴표로 감쌈)를 SRC에 담음.
- `pd.read_csv(SRC)` : pandas의 함수. CSV 파일을 읽어 **표(DataFrame, 줄여서 df)** 로 만든다.
- `df.rename(columns={'label': 'potency'})` : 열 이름 바꾸기. **`{ }`** 는 **딕셔너리**(이름표→값 쌍). 여기선 'label→potency'로 바꾸라는 뜻.
- `df.shape` : 괄호가 **없는** 것은 함수가 아니라 **속성(attribute)**. 표의 크기 (행 수, 열 수)를 준다.
- `df.potency` / `df['potency']` : 표에서 **한 열**을 고르기(둘 다 같은 뜻).
- `.value_counts()` : 그 열의 **값별 개수**를 세는 함수(표에 딸린 함수라 **메서드**라고 부름).
- `dict(...)` : 결과를 보기 좋은 딕셔너리 형태로 변환.

### 셀 3 — descriptor 217개 계산 → 전체 Excel 저장
분자를 하나씩 반복하며 descriptor를 계산해 모으고, 표로 만들어 Excel로 저장한다.

In [ ]:
# RDKit 2D descriptor 217종 계산 -> 전체 Excel 저장(메타 + potency + descriptor)
desc_names = [n for n, _ in Descriptors._descList]
print('descriptor', len(desc_names), '종 계산 중... (수 분 소요)')

rows, keep = [], []
for i, smi in enumerate(df['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        continue
    d = Descriptors.CalcMolDescriptors(m)
    rows.append([d.get(n, np.nan) for n in desc_names])
    keep.append(i)
    if (i + 1) % 1000 == 0:
        print('  ', i + 1, '/', len(df))

X = pd.DataFrame(rows, columns=desc_names)
meta = df.iloc[keep][['canonical_smiles', 'inchikey', 'source', 'potency']].reset_index(drop=True)
full = pd.concat([meta, X.reset_index(drop=True)], axis=1)

XLSX = 'data/HSD17B13_1to1_descriptors.xlsx'
full.to_excel(XLSX, index=False)
print('[Excel] 전체 저장:', XLSX, '| shape', full.shape,
      '(메타 4 + descriptor', len(desc_names), ')')

🔎 **코드 뜯어보기 (셀 3)**
- `[n for n, _ in Descriptors._descList]` : **리스트 컴프리헨션** — 목록을 한 줄로 만드는 문법. "`Descriptors._descList`의 각 항목에서 이름 n만 꺼내 리스트로". `_`는 "안 쓸 값" 자리표시.
- `rows, keep = [], []` : **`[ ]`** 는 **리스트**(순서 있는 빈 상자). 두 변수를 동시에 빈 리스트로 초기화.
- `for i, smi in enumerate(목록):` : **for 반복문** — 목록을 하나씩 꺼내 반복. **enumerate**는 **번호 i**와 **값 smi**를 함께 준다.
- `Chem.MolFromSmiles(...)` : SMILES 글자를 **실제 분자**로 변환(실패하면 `None`).
- `if m is None: continue` : 변환 실패면 **continue**(이번 반복 건너뛰고 다음으로).
- `Descriptors.CalcMolDescriptors(m)` : 분자 하나의 descriptor 217개를 **딕셔너리**로 계산.
- `rows.append([...])` : **.append(x)** = 리스트 끝에 x를 추가.
- `pd.DataFrame(rows, columns=...)` : 값 목록 rows를 **표**로 만들기(columns=열 이름).
- `df.iloc[keep]` : **iloc**는 '행 번호로 고르기'. keep에 담긴 번호의 행만 선택.
- `pd.concat([A, B], axis=1)` : 표 A와 B를 **좌우로**(axis=1=열 방향) 이어붙이기.
- `full.to_excel(경로)` : 표를 **엑셀 파일로 저장**.
- `'%d / %d' % (a, b)` : **문자열 포맷** — `%d` 자리에 뒤 값들을 끼워 넣기.

### 셀 4 — WEKA에 넣을 CSV로 정제
숫자 열만 남기고, 결측을 없애고, 정답을 맨 끝 열에 두어 CSV로 저장한다.

In [ ]:
# ===== WEKA용 CSV 정제 =====
# 규칙: 숫자형 descriptor만 + 클래스(potency)는 '마지막 열' + 문자열 컬럼 삭제 + 결측 없음
compound_info = full[['canonical_smiles', 'inchikey', 'source', 'potency']]
descriptor_data = full[desc_names]
print('[1] descriptor 데이터 shape:', descriptor_data.shape)

# 1) 숫자로 변환 안 되는(문자/이상값) 컬럼 제거
def remove_invalid_descriptors(data):
    invalid = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col], errors='raise')
        except Exception:
            invalid.append(col)
    print('[2] 숫자변환 실패(문자/이상) 컬럼 제거:', len(invalid), '개')
    return data.drop(columns=invalid)

desc_num = remove_invalid_descriptors(descriptor_data).apply(pd.to_numeric)

# 2) inf -> NaN 후, 빈 값 있는 '행' 제거 (모든 descriptor 열은 유지)
desc_num = desc_num.replace([np.inf, -np.inf], np.nan)
n_nan_cell = int(desc_num.isna().sum().sum())
nan_row_mask = desc_num.isna().any(axis=1)
print('[3] 결측/inf 셀', n_nan_cell, '개 -> 결측 포함 행', int(nan_row_mask.sum()), '개 제거')
desc_num = desc_num[~nan_row_mask].reset_index(drop=True)
potency = compound_info['potency'][~nan_row_mask.values].reset_index(drop=True)

# 3) 정답(potency)을 맨 끝 열로 붙여 저장
weka_df = desc_num.copy()
weka_df['potency'] = potency.values
print('[4] WEKA용 shape (descriptor + potency):', weka_df.shape)
print('    마지막 열:', weka_df.columns[-1], '| 클래스 분포:', dict(weka_df.potency.value_counts()))

CSV = 'data/HSD17B13_1to1_descriptors_weka.csv'
weka_df.to_csv(CSV, index=False)
print('[5] WEKA용 CSV 저장 완료:', CSV)

🔎 **코드 뜯어보기 (셀 4)**
- `full[['a','b']]` : 대괄호 안에 **리스트**로 여러 열 이름을 주면 그 **여러 열**을 한 번에 고른다.
- `def remove_invalid_descriptors(data):` : **def**는 **함수 정의**. `data`는 함수가 받는 **입력(매개변수)**. 아래 들여쓴 부분이 함수 몸통.
- `try: ... except Exception: ...` : **try**에서 코드를 시도하고, 오류가 나면 **except** 쪽을 실행(멈추지 않고 대처). 여기선 '숫자 변환이 안 되면 그 열을 버릴 목록에 넣기'.
- `return data.drop(columns=invalid)` : **return**은 함수의 **결과를 돌려줌**. `.drop(columns=...)`는 지정한 열들을 **버린** 표.
- `.apply(pd.to_numeric)` : 각 열에 `pd.to_numeric`(숫자 변환)을 **일괄 적용**.
- `.replace([np.inf, -np.inf], np.nan)` : 무한대(inf)를 **빈 값(NaN)** 으로 바꿈.
- `.isna()` : 각 칸이 빈 값인지 True/False로. `.sum().sum()` : True 개수를 다 합쳐 총 결측 수.
- `.any(axis=1)` : 각 **행**에 빈 값이 하나라도 있으면 True (axis=1=행 방향).
- `~nan_row_mask` : **`~`** 는 **부정**(True↔False 뒤집기). `df[~mask]` = 'mask가 아닌 행만' 고르기(**불리언 인덱싱**).
- `.copy()` : 표를 **복사**(원본 보호). `weka_df['potency'] = ...` : **새 열**을 추가(맨 끝에 붙음).
- `.to_csv(경로, index=False)` : CSV로 저장(index=False=행 번호는 저장 안 함).